# FinAdvisor Chatbot: Domain-Specific Chatbot Using Transformer Models

This notebook implements a financial advisor chatbot that can answer questions about stocks, companies, and market data using fine-tuned transformer models.

## Project Overview
- **Domain**: Financial advisory for stock market queries
- **Model**: GPT-2 fine-tuned for generative QA
- **Data**: Synthetic conversational dataset based on S&P 500 company data
- **Interface**: Streamlit web application

## Table of Contents
1. [Data Preparation](#Data-Preparation)
2. [Model Fine-tuning](#Model-Fine-tuning)
3. [Evaluation](#Evaluation)
4. [Deployment](#Deployment)

In [1]:
# Install required packages
!pip install transformers torch datasets pandas numpy scikit-learn nltk streamlit gradio
!pip install accelerate -U
!pip install evaluate rouge-score

In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import Dataset
import json
import random
from sklearn.model_selection import train_test_split
import evaluate
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/ngabotech/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Data Preparation

First, let's load and explore the available datasets to create a synthetic conversational dataset.

In [3]:
# Load datasets
securities_df = pd.read_csv('../data/securities.csv')
print("Securities dataset shape:", securities_df.shape)
print(securities_df.head())

# Note: fundamentals.csv and prices.csv are too large to load directly
# We'll create synthetic data based on securities info

Securities dataset shape: (505, 8)
  Ticker symbol             Security SEC filings             GICS Sector  \
0           MMM           3M Company     reports             Industrials   
1           ABT  Abbott Laboratories     reports             Health Care   
2          ABBV               AbbVie     reports             Health Care   
3           ACN        Accenture plc     reports  Information Technology   
4          ATVI  Activision Blizzard     reports  Information Technology   

                GICS Sub Industry   Address of Headquarters Date first added  \
0        Industrial Conglomerates       St. Paul, Minnesota              NaN   
1           Health Care Equipment   North Chicago, Illinois       1964-03-31   
2                 Pharmaceuticals   North Chicago, Illinois       2012-12-31   
3  IT Consulting & Other Services           Dublin, Ireland       2011-07-06   
4     Home Entertainment Software  Santa Monica, California       2015-08-31   

       CIK  
0    66740  
1

In [5]:
# Create synthetic conversational dataset
def create_synthetic_dataset(securities_df, num_samples=5000):
    conversations = []
    
    # Question templates
    question_templates = [
        "What is the sector of {company}?",
        "Tell me about {company} stock.",
        "What industry is {company} in?",
        "Where is {company} headquartered?",
        "What does {company} do?",
        "Give me information about {ticker}.",
        "What sector does {ticker} belong to?",
        "Is {company} in the {sector} sector?",
        "Tell me the headquarters location of {company}.",
        "What is {ticker}'s industry?"
    ]
    
    # Answer templates
    answer_templates = [
        "{company} is in the {sector} sector and {sub_industry} industry. It's headquartered in {location}.",
        "{company} ({ticker}) operates in the {sector} sector, specifically in {sub_industry}. The company is based in {location}.",
        "The company {company} is part of the {sector} sector and works in {sub_industry}. Headquarters: {location}.",
        "{ticker} belongs to {company}, which is in the {sector} sector and {sub_industry} industry, located in {location}.",
        "{company} is a {sector} company specializing in {sub_industry}, with headquarters in {location}."
    ]
    
    for _ in range(num_samples):
        company = securities_df.sample(1).iloc[0]
        
        # Select random question template
        question_template = random.choice(question_templates)
        
        # Fill in question
        if '{ticker}' in question_template:
            question = question_template.format(ticker=company['Ticker symbol'])
        elif '{sector}' in question_template:
            question = question_template.format(
                company=company['Security'],
                sector=company['GICS Sector']
            )
        else:
            question = question_template.format(company=company['Security'])
            
        # Select random answer template
        answer_template = random.choice(answer_templates)
        answer = answer_template.format(
            company=company['Security'],
            ticker=company['Ticker symbol'],
            sector=company['GICS Sector'],
            sub_industry=company['GICS Sub Industry'],
            location=company['Address of Headquarters']
        )
        
        conversations.append({
            'question': question,
            'answer': answer
        })
    
    return conversations

# Generate dataset
conversations = create_synthetic_dataset(securities_df, 10000)
print(f"Generated {len(conversations)} conversation pairs")
print("Sample conversation:")
print("Q:", conversations[0]['question'])
print("A:", conversations[0]['answer'])

Generated 10000 conversation pairs
Sample conversation:
Q: Where is Verizon Communications headquartered?
A: Verizon Communications (VZ) operates in the Telecommunications Services sector, specifically in Integrated Telecommunications Services. The company is based in New York, New York.


In [6]:
# Save synthetic dataset
with open('../data/processed/conversations.json', 'w') as f:
    json.dump(conversations, f, indent=2)

# Convert to HuggingFace dataset format
train_data, test_data = train_test_split(conversations, test_size=0.1, random_state=42)

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 9000
Test dataset size: 1000


## Model Fine-tuning

Now we'll fine-tune a GPT-2 model on our conversational dataset.

In [ ]:
# Load tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("distilgpt2")

# Tokenization function
def tokenize_function(examples):
    texts = [f"Question: {q}\nAnswer: {a}" for q, a in zip(examples['question'], examples['answer'])]
    return tokenizer(texts, truncation=True, padding='max_length', max_length=512)

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask'])
tokenized_test.set_format(type='torch', columns=['input_ids', 'attention_mask'])

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir='../models/finadvisor_gpt2',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='../models/logs',
    logging_steps=100,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

# Train the model
trainer.train()

# Save the model
trainer.save_model('../models/finadvisor_gpt2')
tokenizer.save_pretrained('../models/finadvisor_gpt2')

## Evaluation

Evaluate the model using various NLP metrics.

In [ ]:
# Load evaluation metrics
bleu = evaluate.load('bleu')
rouge = evaluate.load('rouge')

# Function to generate answers
def generate_answer(question, model, tokenizer, max_length=100):
    input_text = f"Question: {question}\nAnswer:"
    input_ids = tokenizer.encode(input_text, return_tensors='pt')
    
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=max_length,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            early_stopping=True
        )
    
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    # Extract answer part
    answer = generated_text.split('Answer:')[-1].strip()
    return answer

# Evaluate on test set
predictions = []
references = []

for item in test_data[:100]:  # Evaluate on subset for speed
    pred = generate_answer(item['question'], model, tokenizer)
    predictions.append(pred)
    references.append([item['answer']])

# Calculate BLEU
bleu_score = bleu.compute(predictions=predictions, references=references)
print(f"BLEU Score: {bleu_score['bleu']}")

# Calculate ROUGE
rouge_score = rouge.compute(predictions=predictions, references=[ref[0] for ref in references])
print(f"ROUGE Scores: {rouge_score}")

# Calculate perplexity
def calculate_perplexity(model, tokenizer, dataset):
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for item in dataset:
            input_ids = item['input_ids'].unsqueeze(0)
            labels = input_ids.clone()
            
            outputs = model(input_ids, labels=labels)
            loss = outputs.loss
            
            total_loss += loss.item() * input_ids.size(1)
            total_tokens += input_ids.size(1)
    
    perplexity = torch.exp(torch.tensor(total_loss / total_tokens))
    return perplexity.item()

perplexity = calculate_perplexity(model, tokenizer, tokenized_test)
print(f"Perplexity: {perplexity}")

## Chatbot Interface

Create a simple chatbot interface using Gradio.

In [ ]:
# Create chatbot application
chatbot_code = '''
import gradio as gr
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

# Load model and tokenizer
model_path = "../models/finadvisor_gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path)
model.eval()

def generate_answer(question):
    input_text = f"Question: {question}\nAnswer:"
    input_ids = tokenizer.encode(input_text, return_tensors='pt')
    
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=150,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            early_stopping=True,
            temperature=0.7,
            top_p=0.9
        )
    
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = generated_text.split('Answer:')[-1].strip()
    return answer

def chatbot_response(message, history):
    response = generate_answer(message)
    return response

# Create Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="FinAdvisor Chatbot",
    description="Ask me questions about stocks and companies!",
    examples=[
        "What is Apple's sector?",
        "Tell me about Microsoft",
        "Where is Google headquartered?",
        "What industry is Amazon in?"
    ]
)

if __name__ == "__main__":
    iface.launch()
'''

# Save chatbot code
with open('../app/chatbot.py', 'w') as f:
    f.write(chatbot_code)

print("Chatbot application created at ../app/chatbot.py")

## Summary

This notebook demonstrates the complete pipeline for creating a domain-specific chatbot:

1. **Data Preparation**: Created synthetic conversational dataset from S&P 500 securities data
2. **Model Training**: Fine-tuned GPT-2 model for generative question answering
3. **Evaluation**: Assessed performance using BLEU, ROUGE, and perplexity metrics
4. **Deployment**: Created a Gradio-based chatbot interface

### Key Results:
- BLEU Score: {bleu_score}
- ROUGE Scores: {rouge_score}
- Perplexity: {perplexity}

### Next Steps:
- Run the chatbot: `python ../app/chatbot.py`
- Further hyperparameter tuning
- Add more diverse training data
- Implement conversation memory